### Defining a function that converts RA/DEC from degrees to Mollweide-friendly radians ###

This is done because when we plot a molleweide projection later the function responsible for it only takes in radians as arguments.

Note: this is pretty counter-intuitive because the function is after all: displaying our output in degrees 🤣.

In [ ]:
def prepare_coords(ra_deg, dec_deg):
    """Convert RA/DEC from degrees to Mollweide-friendly radians."""
    ra = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)
    ra = np.remainder(ra + 2*np.pi, 2*np.pi)
    ra[ra > np.pi] -= 2*np.pi
    return -ra, dec

### Making a Plot of ZTF SNIa host galaxy locations vs. the bright and dark galaxies from DESI ###

It looks like there are not host galaxy info given as part of the ZTF data release. But we can still plot SNIa locations and see what is going on. 

In [ ]:
# === Load DESI Bright ===
bright = Table.read("DES5YR_DESI_data/desi_galaxy_metadata.fits")
ra_bright, dec_bright = prepare_coords(bright['RA'], bright['DEC'])

# === Load DESI Dark ===
dark = Table.read("DES5YR_DESI_data/desi_galaxy_metadata_dark.fits")
ra_dark, dec_dark = prepare_coords(dark['RA'], dark['DEC'])

# === Load SNIa Metadata CSV ===
sn_df = pd.read_csv("ZTF_DESI_data/ZTF_snia_with_hosts_AND_SALT2.csv")

# Filter out invalid host coordinates
valid_hosts = (sn_df['ra_host'] != -999) & (sn_df['dec_host'] != -999)
host_ra = sn_df.loc[valid_hosts, 'ra_host'].values
host_dec = sn_df.loc[valid_hosts, 'dec_host'].values
ra_host, dec_host = prepare_coords(host_ra, host_dec)

# === Plot ===
plt.figure(figsize=(13, 6))
ax = plt.subplot(111, projection='mollweide')

# Flatten arrays for hexbin
x_bright, y_bright = ra_bright.flatten(), dec_bright.flatten()
x_dark, y_dark = ra_dark.flatten(), dec_dark.flatten()
x_host, y_host = ra_host.flatten(), dec_host.flatten()

# Hexbin with LogNorm for density contrast

# DESI Bright (density)
hb1 = ax.hexbin(x_bright, y_bright, gridsize=500, cmap='Greens',
                mincnt=1, linewidths=0, norm=LogNorm())

# DESI Dark (density)
hb2 = ax.hexbin(x_dark, y_dark, gridsize=500, cmap='Greys',
                mincnt=1, linewidths=0, norm=LogNorm())

# SNIa Hosts (scatter instead of density)
ax.scatter(x_host, y_host, s=10, color='crimson', alpha=0.6, label='SNIa Host Galaxies w/t SNIa mu')

# Legend using dummy patches
legend_elements = [
    Patch(facecolor='green', label='DESI Bright Galaxies', alpha=1),
    Patch(facecolor='gray', label='DESI Dark Galaxies', alpha=1),
    Patch(facecolor='crimson', label='SNIa Host Galaxies w/t SNIa mu', alpha=1)
]
ax.legend(handles=legend_elements, loc='upper right')

# Plot settings
ax.set_title('ZTF Surveys & SNIa Host Galaxies (Mollweide)', pad=20)
ax.set_xlabel('Right Ascension')
ax.set_ylabel('Declination')
ax.grid(True)
plt.show()

In [ ]:
# ----------------------------
# Paths (adjust if needed)
# ----------------------------
DESI_BRIGHT_FITS = "DES5YR_DESI_data/desi_galaxy_metadata.fits"
DESI_DARK_FITS   = "DES5YR_DESI_data/desi_galaxy_metadata_dark.fits"
DES_SN_CSV       = "ZTF_DESI_data/ZTF_snia_data.csv"
MATCHED_CSV      = "ZTF_DESI_data/ZTF_snia_match_DESI_hostgal_Z_filtered.csv"

# ----------------------------
# Helper
# ----------------------------
def clean_numeric(series, finite_only=True):
    arr = pd.to_numeric(series, errors="coerce").to_numpy()
    if finite_only:
        arr = arr[np.isfinite(arr)]
    return arr

# ----------------------------
# Load DESI galaxy redshifts
# ----------------------------
desi_bright = Table.read(DESI_BRIGHT_FITS, hdu=1).to_pandas()
desi_dark   = Table.read(DESI_DARK_FITS,   hdu=1).to_pandas()

Z_bright = clean_numeric(desi_bright.get("Z", pd.Series([])))
Z_dark   = clean_numeric(desi_dark.get("Z", pd.Series([])))

# ----------------------------
# Load ZTF SNIa
# ----------------------------
des_sn = pd.read_csv(DES_SN_CSV)
zHEL_all = clean_numeric(des_sn.get("redshift", pd.Series([])))

# ----------------------------
# Load matched SNe
# ----------------------------
matched = pd.read_csv(MATCHED_CSV)
zHEL_matched = clean_numeric(matched.get("redshift", pd.Series([])))

# ----------------------------
# Plot: all distributions together
# ----------------------------
zmin, zmax = 0.0, 1.0
bins = np.linspace(zmin, zmax, 100)

plt.figure(figsize=(10, 6))
plt.hist(Z_bright, bins=bins, alpha=0.4, label="DESI Bright", color = "green", log=True)
plt.hist(Z_dark, bins=bins, alpha=0.4, label="DESI Dark", color = "black", log=True)
plt.hist(zHEL_all, bins=bins, alpha=0.6, label="ZTF SNe Ia", color = "red", log=True)
plt.hist(zHEL_matched, bins=bins, label="Matched ZTF SNe Ia", color = "black", log=True)

plt.xlabel("Redshift z")
plt.ylabel("Count (log scale)")
plt.title("Redshift Distributions: DESI and DES-SN5YR")
plt.legend(
    loc="upper left", 
    bbox_to_anchor=(1.02, 1),   # push legend just outside to the right
    borderaxespad=0
)
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# Build shared bins from ALL redshifts (no cutoffs)
# ----------------------------
arrays = [Z_bright, Z_dark, zHEL_all, zHEL_matched]
nonempty = [a for a in arrays if a.size > 0]
all_z = np.concatenate(nonempty) if nonempty else np.array([])

# Safety: if everything's empty, skip; otherwise compute edges from data
if all_z.size == 0:
    raise ValueError("No redshift values found after cleaning.")

# Option A (recommended): data-driven binning (Freedman–Diaconis)
bins = np.histogram_bin_edges(all_z, bins="fd")
print(len(bins))
bins = np.linspace(all_z.min(), all_z.max(), 500)


# Fallback: if FD returns <2 edges (e.g., tiny sample), use a simple linspace
if bins.size < 2:
    zmin, zmax = float(all_z.min()), float(all_z.max())
    if zmin == zmax:
        zmax = zmin + 1e-6
    bins = np.linspace(zmin, zmax, 100)

plt.figure(figsize=(10, 6))
plt.hist(Z_bright, bins=bins, alpha=0.4, label="DESI Bright", color="green", log=True)
plt.hist(Z_dark,   bins=bins, alpha=0.4, label="DESI Dark",   color="black", log=True)
plt.hist(zHEL_all, bins=bins, alpha=0.6, label="DES-SN5YR SNe Ia", color="red", log=True)
plt.hist(zHEL_matched, bins=bins, label="Matched SNe Ia", color="black", log=True)

plt.xlabel("Redshift z")
plt.ylabel("Count (log scale)")
plt.tight_layout()
plt.show()